# Library loaded.

In [34]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import re

warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning) # Ignore unnecessary warnings from BeautifulSoup

nltk.download('stopwords')
nltk.download('wordnet') 

nlp = spacy.load("en_core_web_sm", disable=["parser", "senter"]) # model for proper noun detection

print("Libraries loaded successfully.")

[nltk_data] Downloading package stopwords to /home/jaee/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jaee/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Libraries loaded successfully.


## Data Loaded

In [35]:
# -------------------------------------------------------------------
# UPDATE THIS PATH to point to your full ~6000 page dataset
# -------------------------------------------------------------------
DATA_PATH = "../data/english_pages_metadata_clean_with_labels.csv"

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df['full_text'] = df['full_text'].fillna('').astype(str)

print(f"Loaded {len(df)} pages")
print(f"Columns: {list(df.columns)}")

# Text length statistics
print(f"\nText length statistics (characters):")
print(df['full_text'].str.len().describe())

df.head(3)

Loaded 596 pages
Columns: ['page_id', 'assigned_to', 'manual_label', 'manual_label_clean', 'manual_label_final', 'full_text', '_merge']

Text length statistics (characters):
count      596.000000
mean      6535.971477
std       9110.613565
min        294.000000
25%       1748.500000
50%       3446.500000
75%       7771.000000
max      75582.000000
Name: full_text, dtype: float64


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,_merge
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,both
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,both
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,both


## Label Checked

In [36]:
# Load labels if available (for downstream evaluation in Notebook 2)
LABELS_PATH = "../data/english_pages_metadata_clean_with_labels.csv"

if os.path.exists(LABELS_PATH):
    labels_df = pd.read_csv(LABELS_PATH)
    labels_subset = labels_df[['page_id', 'manual_label_final']].copy()
    df = pd.merge(df, labels_subset, on='page_id', how='left')
    print(f"Labels merged. Distribution:")
    print(df['manual_label_final'].value_counts())
else:
    print(f"Labels file not found. Proceeding without labels.")

Labels merged. Distribution:


KeyError: 'manual_label_final'

## Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Remove \n, \r, \t
    text = re.sub(r'[\n\r\t]+', ' ', text)
    
    # Remove URLs and emails
    text = re.sub(r'(?:https?://|ftp://|www\.)\S+|mailto:\S+|tel:\S+', '', text)
    text = re.sub(r'\b[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}\b', '', text)

    # Remove HTML tags like <body>, but first remove script/style content
    soup = BeautifulSoup(text, 'html.parser')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    text = soup.get_text(separator=' ')

    # Build proper noun set
    # proper_nouns = build_proper_noun_set_spacy(text)

    # Split camelCases
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

    # Remove proper nouns
    # text = ' '.join(w for w in text.split() if w.lower() not in proper_nouns)

    # Remove separators such as _, -, /, |, ...
    text = re.sub(r'[_\-/|]', ' ', text)
    
    # Lowercase
    text = text.lower()
        
    # Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]', ' ', text)

    # Normalize repeated characters like 'yessss', 'noooooo', ...
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip() # remove multiple spaces

    # Tokenize and filter with stop words and lemmatization
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if len(t) >= 3
              and len(t) <= 20 # Remove very long words
              and t not in stop_words] # Keep 3 letter words like api, cpu, ...
    
    return tokens

## Input data preparation

In [ ]:
# Prepare text list used by all models
texts = df['full_text'].apply(preprocess).tolist()
print(texts[0][:100])
print(f"Prepared {len(texts)} documents for embedding.")

['fall', 'arrest', 'work', 'positioning', 'harness', 'allied', 'safety', 'equipment', 'pvt', 'ltd', 'skip', 'content', 'sport', 'professional', 'search', 'search', 'enquire', 'activity', 'industry', 'rope', 'access', 'confined', 'space', 'facade', 'cleaning', 'wind', 'mill', 'bird', 'netting', 'tree', 'care', 'energy', 'network', 'framing', 'roofing', 'stunt', 'stunt', 'vest', 'aerial', 'acrobatics', 'corset', 'waist', 'chest', 'strap', 'accessory', 'adventure', 'park', 'harness', 'rescue', 'building', 'rescue', 'technical', 'rescue', 'ski', 'lift', 'rescue', 'site', 'rescue', 'military', 'intervention', 'slithering', 'brand', 'industry', 'petzl', 'tractel', 'portable', 'winch', 'tracer', 'mode', 'rescue', 'hwi', 'fast', 'rope', 'rope', 'petzl', 'sterling', 'teufelberger', 'special', 'rope', 'product', 'industry', 'helmet', 'headlamp', 'harness', 'connector', 'descender', 'rope', 'clamp', 'pulley', 'anchor', 'lanyard', 'rope', 'fixed', 'line', 'fall', 'protection', 'temporary', 'line',

## all-MiniLM-L6-v2 loaded

In [ ]:
from sentence_transformers import SentenceTransformer

mini_model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Model loaded: all-MiniLM-L6-v2")
print(f"Max sequence length: {mini_model.max_seq_length}")
print(f"Embedding dimension: {mini_model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: all-MiniLM-L6-v2
Max sequence length: 256
Embedding dimension: 384


## Generate Embedding using all-MiniLM-L6-v2

In [ ]:
print(f"Generating MiniLM embeddings for {len(texts)} documents...")
start_time = time.time()

mini_embeddings = mini_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f} seconds ({len(texts)/elapsed:.1f} docs/sec)")
print(f"Embedding matrix shape: {mini_embeddings.shape}")

Generating MiniLM embeddings for 596 documents...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]


Done in 0.2 seconds (2732.0 docs/sec)
Embedding matrix shape: (596, 384)


## Embedding by all-MiniLM-L6-v2 saved

In [ ]:
np.save('minilm_embeddings.npy', mini_embeddings)
print(f"Saved minilm_embeddings.npy, shape {mini_embeddings.shape}")

Saved minilm_embeddings.npy, shape (596, 384)


## OpenAI text-embedding-3-small Loaded

In [ ]:
from openai import OpenAI
import tiktoken
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

OPENAI_MODEL = "text-embedding-3-small"

encoding = tiktoken.get_encoding("cl100k_base")
MAX_TOKENS = 8000

print(f"OpenAI client initialized. Model: {OPENAI_MODEL}")

OpenAI client initialized. Model: text-embedding-3-small


In [ ]:
print(texts[0][:100])

['fall', 'arrest', 'work', 'positioning', 'harness', 'allied', 'safety', 'equipment', 'pvt', 'ltd', 'skip', 'content', 'sport', 'professional', 'search', 'search', 'enquire', 'activity', 'industry', 'rope', 'access', 'confined', 'space', 'facade', 'cleaning', 'wind', 'mill', 'bird', 'netting', 'tree', 'care', 'energy', 'network', 'framing', 'roofing', 'stunt', 'stunt', 'vest', 'aerial', 'acrobatics', 'corset', 'waist', 'chest', 'strap', 'accessory', 'adventure', 'park', 'harness', 'rescue', 'building', 'rescue', 'technical', 'rescue', 'ski', 'lift', 'rescue', 'site', 'rescue', 'military', 'intervention', 'slithering', 'brand', 'industry', 'petzl', 'tractel', 'portable', 'winch', 'tracer', 'mode', 'rescue', 'hwi', 'fast', 'rope', 'rope', 'petzl', 'sterling', 'teufelberger', 'special', 'rope', 'product', 'industry', 'helmet', 'headlamp', 'harness', 'connector', 'descender', 'rope', 'clamp', 'pulley', 'anchor', 'lanyard', 'rope', 'fixed', 'line', 'fall', 'protection', 'temporary', 'line',

## Embedding generated by text-embedding-3-small

In [ ]:
def truncate_text(text, max_tokens=MAX_TOKENS):
    """Truncate text to fit within token limit."""
    tokens = encoding.encode(text)
    if len(tokens) > max_tokens:
        return encoding.decode(tokens[:max_tokens])
    return text

def get_openai_embeddings(texts, client, model=OPENAI_MODEL, batch_size=100):
    """Generate OpenAI embeddings in batches."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="OpenAI batches"):
        batch = texts[i : i + batch_size]
        batch = [t if t.strip() else " " for t in batch]
        response = client.embeddings.create(model=model, input=batch)
        all_embeddings.extend([item.embedding for item in response.data])
    return np.array(all_embeddings)

texts_strings = [" ".join(t) for t in texts]
oai_texts = [truncate_text(t) for t in texts_strings]

print(f"Generating OpenAI embeddings for {len(oai_texts)} documents...")
print(f"Estimated cost: ~${len(oai_texts) * 500 * 0.02 / 1_000_000:.2f}")
start_time = time.time()

oai_embeddings = get_openai_embeddings(oai_texts, client)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f} seconds")
print(f"Embedding matrix shape: {oai_embeddings.shape}")

Generating OpenAI embeddings for 596 documents...
Estimated cost: ~$0.01


OpenAI batches: 100%|██████████| 6/6 [00:13<00:00,  2.18s/it]


Done in 13.1 seconds
Embedding matrix shape: (596, 1536)


## Embedding by text-embedding-3-small saved

In [ ]:
np.save('openai_embeddings.npy', oai_embeddings)
print(f"Saved openai_embeddings.npy, shape {oai_embeddings.shape}")

Saved openai_embeddings.npy, shape (596, 1536)


## Load bge-small-en-v1.5 model, and generate embeddings
- bge-small-en-v1.5 model has 384D, which is the same dimension as all-MiniLM-L6-v2.
- bge-small-en-v1.5 model has clustering score of about 50, while all-MiniLM-L6-v2 has about 42.
- bge-small-en-v1.5 model has MTEB average score of about 61, while all-MiniLM-L6-v2 has about 56.
- Thus, it is a good model to compare against all-MiniLM-L6-v2, since it has higher score while having same dimensions of 384D and 512 max tokens.
- This model has more modern architecture. We are expecting this model to be a bit slower, but more accurate and better in quality.

In [ ]:
bge_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

print(f"Generating BGE embeddings for {len(texts)} documents...")
start_time = time.time()

bge_embeddings = bge_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f} seconds ({len(texts)/elapsed:.1f} docs/sec)")
print(f"Embedding matrix shape: {bge_embeddings.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating BGE embeddings for 596 documents...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]


Done in 0.3 seconds (1843.8 docs/sec)
Embedding matrix shape: (596, 384)


## Save BGE embeddings

In [37]:
np.save('bge_embeddings.npy', bge_embeddings)
print(f"Saved bge_embeddings.npy, shape {bge_embeddings.shape}")

Saved bge_embeddings.npy, shape (596, 384)
